In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/comm_research").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the commodity-research checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [2]:
import pandas as pd
import polars as pl
import numpy as np
import scipy.stats as st

In [3]:
from comm_research.infra.lake.parquet import scan_parquet
from comm_research.infra.tools.vis import *

In [4]:
## global vars

ACTUALS_DIR = PROJECT_ROOT / "data/clean/weather/era5/t_2m/socal"  
FCAST_DIR = PROJECT_ROOT / "data/clean/weather/gefs/v12/t_2m/socal/4wk"  
GRID_ID = 0

In [5]:
def load_data(data_dir, grid_id=None):
    grid_id = GRID_ID if grid_id is None else grid_id
    paths = sorted(Path(data_dir).glob(f"*_{grid_id}.csv"))
    if not paths:
        raise FileNotFoundError(f"No CSV for grid ID {grid_id} in {data_dir}")
    if len(paths) != 1:
        raise ValueError(f"Multiple CSVs for grid ID {grid_id} in {data_dir}: {paths}")
    return (pd.read_csv(paths[0], parse_dates=["timestamp"])
            .sort_values("timestamp")
            .reset_index(drop=True))

def utc(value):
        value = pd.Timestamp(value)
        return (
            value.tz_localize("UTC")
            if value.tzinfo is None
            else value.tz_convert("UTC")
        )


In [6]:
actuals = load_data(ACTUALS_DIR)
forecasts = load_data(FCAST_DIR)

In [19]:
actuals

,grid_id,timestamp,lat,lon,2m_mean,2m_max_6h,2m_min_6h,n_samples,valid_end,sampling_hours,units,aggregation
0,0,2000-01-01 00:00:00+00:00,33.75,-121.0,285.803528,286.418701,285.173096,28,2000-01-08T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
1,0,2000-01-02 00:00:00+00:00,33.75,-121.0,285.814026,286.418701,285.173096,28,2000-01-09T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
2,0,2000-01-03 00:00:00+00:00,33.75,-121.0,285.816598,286.418701,285.171387,28,2000-01-10T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
3,0,2000-01-04 00:00:00+00:00,33.75,-121.0,285.853315,286.418701,285.171387,28,2000-01-11T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
4,0,2000-01-05 00:00:00+00:00,33.75,-121.0,285.757520,286.418701,284.912109,28,2000-01-12T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
...,...,...,...,...,...,...,...,...,...,...,...,...
9727,0,2026-09-20 00:00:00+00:00,33.75,-121.0,291.305545,292.298248,290.290894,28,2026-09-27T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
9728,0,2026-09-21 00:00:00+00:00,33.75,-121.0,291.317571,292.298248,290.290894,28,2026-09-28T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
9729,0,2026-09-22 00:00:00+00:00,33.75,-121.0,291.314440,292.298248,290.133789,28,2026-09-29T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples
9730,0,2026-09-23 00:00:00+00:00,33.75,-121.0,291.349196,292.298248,290.133789,28,2026-09-30T00:00:00.000000+0000,6,K,mean_of_6h_instantaneous_samples


In [7]:
paired = forecasts.merge(
    actuals[["timestamp", "lat", "lon", "grid_id", "2m_mean", "n_samples"]].rename(
        columns={"2m_mean": "actual_mean_k", "n_samples": "actual_n_samples"}
    ),
    on=["timestamp", "lat", "lon"],
    how="left",
    validate="one_to_one",
)


In [8]:
paired

,timestamp,lat,lon,initialization_date,valid_end,model,lead_days,units,sampling_hours,aggregation,...,perturbed_4,perturbed_5,perturbed_6,perturbed_7,perturbed_8,perturbed_9,perturbed_10,grid_id,actual_mean_k,actual_n_samples
0,2000-01-31 00:00:00+00:00,33.75,-121.0,2000-01-05,2000-02-07T00:00:00.000000+0000,gefs_v12,26,K,6,mean_of_6h_instantaneous_samples,...,285.258578,284.929291,286.732941,284.715753,285.924162,285.891999,284.760122,0.0,286.790588,28.0
1,2000-02-07 00:00:00+00:00,33.75,-121.0,2000-01-12,2000-02-14T00:00:00.000000+0000,gefs_v12,26,K,6,mean_of_6h_instantaneous_samples,...,282.682206,286.349945,281.953961,285.452854,285.586802,284.338841,284.232976,0.0,286.505912,28.0
2,2000-02-14 00:00:00+00:00,33.75,-121.0,2000-01-19,2000-02-21T00:00:00.000000+0000,gefs_v12,26,K,6,mean_of_6h_instantaneous_samples,...,286.521798,283.974717,285.922501,286.560035,287.487089,284.283199,285.692296,0.0,286.680987,28.0
3,2000-02-21 00:00:00+00:00,33.75,-121.0,2000-01-26,2000-02-28T00:00:00.000000+0000,gefs_v12,26,K,6,mean_of_6h_instantaneous_samples,...,285.515799,286.842127,286.772774,286.589423,287.218914,286.571136,285.892110,0.0,286.107640,28.0
4,2000-02-28 00:00:00+00:00,33.75,-121.0,2000-02-02,2000-03-06T00:00:00.000000+0000,gefs_v12,26,K,6,mean_of_6h_instantaneous_samples,...,286.952237,285.535127,286.998141,285.479682,287.009874,284.481645,287.515599,0.0,285.384556,28.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1036,2019-12-23 00:00:00+00:00,33.75,-121.0,2019-11-27,2019-12-30T00:00:00.000000+0000,gefs_v12,26,K,6,mean_of_6h_instantaneous_samples,...,285.954720,286.346521,286.548122,285.844599,287.657719,283.780877,285.554443,0.0,286.314512,28.0
1037,2019-12-30 00:00:00+00:00,33.75,-121.0,2019-12-04,2020-01-06T00:00:00.000000+0000,gefs_v12,26,K,6,mean_of_6h_instantaneous_samples,...,288.373453,286.913338,288.545225,288.476544,285.940411,288.264206,288.880967,0.0,287.000768,28.0
1038,2020-01-06 00:00:00+00:00,33.75,-121.0,2019-12-11,2020-01-13T00:00:00.000000+0000,gefs_v12,26,K,6,mean_of_6h_instantaneous_samples,...,285.968094,287.258714,287.215665,286.075076,287.714700,286.382814,286.299087,0.0,286.479707,28.0
1039,2020-01-13 00:00:00+00:00,33.75,-121.0,2019-12-18,2020-01-20T00:00:00.000000+0000,gefs_v12,26,K,6,mean_of_6h_instantaneous_samples,...,286.063869,285.351276,287.900335,288.334773,287.950711,287.615728,287.072602,0.0,285.656626,28.0


### Constructing Climatological Thresholds

In [9]:
def collect_historical_data(
    df, target_date, window, *, history_years=10, lead_days=26
):
    target = pd.Timestamp(target_date, tz="UTC").normalize()
    w = pd.Timedelta(window)
    ts = pd.to_datetime(df["timestamp"], utc=True)
    vals = pd.to_numeric(df["actual_mean_k"])
    
    eligible = (ts + pd.Timedelta(days=7)).le(target - pd.Timedelta(days=lead_days)) & vals.notna()
    
    mask = pd.Series(False, index=df.index)
    for y in range(target.year - history_years, target.year):
        try:
            center = target.replace(year=y)
        except ValueError:
            center = target.replace(year=y, day=28)
        mask |= ts.between(center - w, center + w)
        
    return vals.loc[mask & eligible].values

In [10]:
def construct_thresholds(data, buckets=5):
    # constructs array of q_i climatology thresholds 
    q_values = [(i / buckets) for i in range(1, buckets + 1)]

    thresholds = np.quantile(
        data,
        q_values,
        method="linear",
    )

    return np.append(thresholds, np.inf)

In [11]:
def construct_cdf(df, target_date, q):
    # constructs cdf vector for model fcasts
    fcast_cols = [col for col in df.columns if 'perturbed' in col]
    fcast_cols.append('control')

    fcast_values = np.array(df.loc[df['timestamp'] == target_date, fcast_cols])

    comparison_matrix = (fcast_values[:, None] < q[None, :])
    return np.mean(comparison_matrix, axis=1)

In [12]:
def construct_actuals(df, target_date, q):
    # ground truth target vector to compare against empirical CDF
    realized_era5 = df.loc[df['timestamp'] == target_date, 'actual_mean_k']
    return (q > realized_era5).astype(int)

### Constructing Model-Ensemble CDFs

### PBC Algorithm

In [13]:
def debiasPP():
    pass

In [14]:
def persistencePP():
    pass

### Isotonic Projection and PBC Corrected Forecast

In [15]:
def isotonic_proj():
    pass

In [16]:
def pbc_forecast():
    pass

In [17]:
members = np.array([10, 15, 20])

In [18]:
members[:, None]

array([[10],
       [15],
       [20]])